## Example 1: Markov Chains

In [ ]:
import quantecon as qe
import numpy as np

# Example 1: Markov Chains
print("Testing quantecon Markov chains:")

# Create a 2-state Markov chain (recession/expansion)
P = np.array([[0.7, 0.3],    # From recession: 70% stay, 30% expand
              [0.2, 0.8]])   # From expansion: 20% recede, 80% stay

mc = qe.MarkovChain(P)
print(f"✓ Transition matrix:\n{mc.P}")
print(f"✓ Number of states: {mc.n}")

# Stationary distribution
stationary = mc.stationary_distributions[0]
print(f"✓ Stationary distribution: {stationary}")

# Simulate the Markov chain
np.random.seed(42)
simulation = mc.simulate(ts_length=20, init=0)
print(f"✓ Simulation (20 periods): {simulation[:10]}...")
# The stationary distribution pi solves pi P = pi: 0.3 pi_0 = 0.2 pi_1 gives (0.4, 0.6)
assert mc.n == 2
assert np.allclose(stationary, [0.4, 0.6]) and np.isclose(stationary.sum(), 1.0)
assert np.allclose(stationary @ P, stationary)
assert len(simulation) == 20 and simulation[0] == 0 and set(simulation) <= {0, 1}

## Example 2: ARMA processes

In [ ]:
import quantecon as qe
import numpy as np

# Example 2: ARMA processes
print("Testing quantecon ARMA processes:")

# Create an ARMA(2,1) process: X_t = 0.8 X_{t-1} - 0.15 X_{t-2} + e_t + 0.4 e_{t-1}
phi = np.array([0.8, -0.15])  # AR coefficients
theta = np.array([0.4])       # MA coefficient

arma = qe.ARMA(phi, theta)
print(f"✓ ARMA(2,1) created")
print(f"  AR coefficients: {arma.phi}")
print(f"  MA coefficients: {arma.theta}")

# Impulse response
impulse = arma.impulse_response(impulse_length=10)
print(f"✓ Impulse response (first 5): {impulse[:5]}")

# Autocovariance
autocov = arma.autocovariance(num_autocov=5)
print(f"✓ Autocovariances: {autocov}")
# psi_0 = 1, psi_1 = 0.8 + 0.4, psi_j = 0.8 psi_{j-1} - 0.15 psi_{j-2} after that
assert np.allclose(impulse[:5], [1.0, 1.2, 0.81, 0.468, 0.2529])
# With unit shock variance, the variance is the sum of the squared impulse responses
long_impulse = arma.impulse_response(impulse_length=200)
assert np.isclose(autocov[0], np.sum(long_impulse ** 2))
assert len(autocov) == 5 and autocov[0] > autocov[1] > autocov[2]

## Example 3: Linear State Space Models

In [ ]:
import quantecon as qe
import numpy as np

# Example 3: Linear State Space Models
print("Testing quantecon LinearStateSpace:")

# AR(1) in state space form: x_{t+1} = 0.8*x_t + w_t, y_t = x_t + v_t
A = np.array([[0.8]])  # State transition
C = np.array([[1.0]])  # State shock
G = np.array([[1.0]])  # Observation matrix
H = np.array([[0.1]])  # Observation noise

lss = qe.LinearStateSpace(A, C, G, H)
print(f"✓ LinearStateSpace created")
print(f"  State dim: {lss.n}, Control dim: {lss.k}, Obs dim: {lss.m}")

# Simulate
np.random.seed(42)
x, y = lss.simulate(ts_length=100)
print(f"✓ Simulated 100 periods")
print(f"  State mean: {np.mean(x):.3f}, std: {np.std(x):.3f}")
print(f"  Obs mean: {np.mean(y):.3f}, std: {np.std(y):.3f}")
assert (lss.n, lss.k, lss.m) == (1, 1, 1)
assert x.shape == (1, 100) and y.shape == (1, 100)
# Stationary state variance of the AR(1): 1 / (1 - 0.8**2); the observation adds H H' = 0.01
mu_x, mu_y, Sigma_x, Sigma_y, Sigma_yx = lss.stationary_distributions()
print(f"✓ Stationary state variance: {Sigma_x[0, 0]:.4f}")
assert np.isclose(Sigma_x[0, 0], 1 / (1 - 0.8 ** 2))
assert np.isclose(Sigma_y[0, 0], Sigma_x[0, 0] + 0.1 ** 2)

## Example 4: Discrete random variables

In [ ]:
import quantecon as qe
import numpy as np

# Example 4: Discrete random variables
print("Testing quantecon DiscreteRV:")

# Create a discrete random variable (probabilities only)
probabilities = np.array([0.1, 0.2, 0.3, 0.25, 0.15])

drv = qe.DiscreteRV(probabilities)
print(f"✓ DiscreteRV created")
print(f"  Probabilities: {probabilities}")

# Draw samples (returns indices 0-4)
np.random.seed(42)
samples = drv.draw(k=1000)
print(f"✓ Drew 1000 samples")
print(f"  Unique values: {np.unique(samples)}")
print(f"  Sample mean: {np.mean(samples):.3f}")
assert len(samples) == 1000 and set(np.unique(samples)) <= {0, 1, 2, 3, 4}
assert np.allclose(drv.Q, np.cumsum(probabilities))   # cumulative probabilities used for drawing
# The expected value is 0*0.1 + 1*0.2 + 2*0.3 + 3*0.25 + 4*0.15 = 2.15
assert abs(np.mean(samples) - 2.15) < 0.2

## Example 5: Simulated dividend growth with a Markov chain

In [ ]:
import quantecon as qe
import numpy as np

# Example 5: Simulated dividend growth with a Markov chain
print("Testing quantecon dividend simulation:")

# Create dividend growth model with 2 states
dividend_growth_rates = np.array([0.95, 1.05])
P_dividends = np.array([[0.8, 0.2], [0.3, 0.7]])

mc_dividends = qe.MarkovChain(P_dividends)
print(f"✓ Dividend growth model:")
print(f"  Growth rates: {dividend_growth_rates}")
print(f"  Stationary dist: {mc_dividends.stationary_distributions[0]}")

# Simulate dividend path
np.random.seed(42)
dividend_states = mc_dividends.simulate(ts_length=50, init=0)
dividend_path = dividend_growth_rates[dividend_states].cumprod()
print(f"✓ Simulated dividends (first 5): {dividend_path[:5]}")
print(f"✓ Average growth: {np.mean(dividend_growth_rates[dividend_states]):.3f}")
# pi P = pi: 0.2 pi_0 = 0.3 pi_1 gives (0.6, 0.4)
pi = mc_dividends.stationary_distributions[0]
assert np.allclose(pi, [0.6, 0.4]) and np.allclose(pi @ P_dividends, pi)
assert dividend_states[0] == 0 and dividend_path[0] == 0.95   # starts in state 0
assert np.allclose(dividend_path, np.cumprod(dividend_growth_rates[dividend_states]))